In [0]:
#Required imports
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Creating tables
data_csv = spark.table("workspace.default.sample_sales_dataset_csv")
data_paraquet = spark.table("workspace.default.sample_sales_dataset_paraquet")

In [0]:
#Printing Schemas
data_csv.printSchema()
data_paraquet.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sales: double (nullable = true)
 |-- discount: long (nullable = true)
 |-- returned: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sales: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- returned: string (nullable = true)



In [0]:
#Show
data_csv.show(5)
data_paraquet.show(5)

#Collect
data_csv.limit(5).collect()
data_paraquet.limit(5).collect()

+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
|order_id|order_date|customer_name|region|       category|quantity|unit_price|  sales|discount|returned|
+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
|       1|2025-06-13|          Eva| South|    Electronics|       2|     49.99|  99.98|       5|     Yes|
|       2|2025-06-23|         John|  West|    Electronics|       9|     49.99| 449.91|       0|     Yes|
|       3|2025-02-25|         John| North|Office Supplies|       4|    199.99| 799.96|       5|      No|
|       4|2025-02-26|          Eva| North|    Electronics|       8|    199.99|1599.92|      15|      No|
|       5|2025-03-13|        Farah| North|    Electronics|       3|     79.99| 239.97|      15|     Yes|
+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
only showing top 5 rows
+--------+----------+----------

[Row(order_id=1, order_date='2025-06-13', customer_name='Eva', region='South', category='Electronics', quantity=2, unit_price=49.99, sales=99.98, discount=5.0, returned='Yes'),
 Row(order_id=2, order_date='2025-06-23', customer_name='John', region='West', category='Electronics', quantity=9, unit_price=49.99, sales=449.91, discount=0.0, returned='Yes'),
 Row(order_id=3, order_date='2025-02-25', customer_name='John', region='North', category='Office Supplies', quantity=4, unit_price=199.99, sales=799.96, discount=5.0, returned='No'),
 Row(order_id=4, order_date='2025-02-26', customer_name='Eva', region='North', category='Electronics', quantity=8, unit_price=199.99, sales=1599.92, discount=15.0, returned='No'),
 Row(order_id=5, order_date='2025-03-13', customer_name='Farah', region='North', category='Electronics', quantity=3, unit_price=79.99, sales=239.97, discount=15.0, returned='Yes')]

In [0]:
#Schema defination can be done this way
#StructField(name,DataType(),nullable?)
schema = StructType([
    StructField("order_id",IntegerType(),False),
    StructField("order_date",StringType(),True),
    StructField("customer_name",StringType(),True),
    StructField("region",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("sales",DoubleType(),True),
    StructField("discount",DoubleType(),True),
    StructField("returned",StringType(),True)
])


In [0]:
#Lazy Evaluation illustration
filtered = data_csv.filter(col("quantity") > 5)

selected = filtered.select(
    "order_id",
    "customer_name",
    "sales"
)

print("Nothing executed yet.")

selected.show()
print("Execution completed when data is shown")

Nothing executed yet.
+--------+-------------+-------+
|order_id|customer_name|  sales|
+--------+-------------+-------+
|       2|         John| 449.91|
|       4|          Eva|1599.92|
|       6|          Eva|1199.94|
|       9|        Farah|2399.92|
|      11|       Ishaan| 699.93|
|      13|        Helen| 479.94|
|      15|        David| 699.93|
|      17|       Ishaan| 1499.9|
|      18|          Eva|   NULL|
|      19|          Eva| 399.92|
|      20|        David|1799.91|
|      21|        Alice| 899.94|
|      24|        David|   NULL|
|      25|        David| 399.92|
|      26|         John| 719.91|
|      28|       Ishaan| 639.92|
|      29|       George|   NULL|
|      30|        Helen|1049.93|
|      31|          Bob|   NULL|
|      32|          Eva| 559.93|
+--------+-------------+-------+
only showing top 20 rows
Execution completed when data is shown


In [0]:
#Execution plan by DAG
selected.explain("formatted")

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet workspace.default.sample_sales_dataset_csv (1)


(1) PhotonScan parquet workspace.default.sample_sales_dataset_csv
Output [4]: [order_id#16400L, customer_name#16402, quantity#16405L, sales#16407]
DictionaryFilters: [(quantity#16405L > 5)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-xj8gb/uc/3cf33afb-8527-46d3-9622-07b25cde4cf2/816a35f3-49c4-4a39-bbd0-46186de42222/__unitystorage/catalogs/d0db1f8f-9bf8-4dc2-ae9b-075c58b38950/tables/6c1eb997-fdba-4db0-a890-a19fae47715e]
ReadSchema: struct<order_id:bigint,customer_name:string,quantity:bigint,sales:double>
RequiredDataFilters: [isnotnull(quantity#16405L), (quantity#16405L > 5)]

(2) PhotonProject
Input [4]: [order_id#16400L, customer_name#16402, quantity#16405L, sales#16407]
Arguments: [order_id#16400L, customer_name#16402, sales#16407]

(3) PhotonColumnarToRow
Input [3]: [order_id#16400L, customer_name#16402, s

In [0]:
#Making a transformation to fileter Electronics from the data
electronics = data_csv.filter(col("category") == "Electronics")
electronics.show(5)

#Filtering products with more than 500 sales and quantity more than 5
popular = electronics.filter((col("sales") > 500) & (col("quantity") > 5))
popular.show(5)

#Filetring orders that have been returned
returned = electronics.filter(col("returned") == "Yes")
returned.show(5)

+--------+----------+-------------+------+-----------+--------+----------+-------+--------+--------+
|order_id|order_date|customer_name|region|   category|quantity|unit_price|  sales|discount|returned|
+--------+----------+-------------+------+-----------+--------+----------+-------+--------+--------+
|       1|2025-06-13|          Eva| South|Electronics|       2|     49.99|  99.98|       5|     Yes|
|       2|2025-06-23|         John|  West|Electronics|       9|     49.99| 449.91|       0|     Yes|
|       4|2025-02-26|          Eva| North|Electronics|       8|    199.99|1599.92|      15|      No|
|       5|2025-03-13|        Farah| North|Electronics|       3|     79.99| 239.97|      15|     Yes|
|       8|2025-01-12|          Eva| North|Electronics|       4|      NULL|   NULL|       0|      No|
+--------+----------+-------------+------+-----------+--------+----------+-------+--------+--------+
only showing top 5 rows
+--------+----------+-------------+------+-----------+--------+----

In [0]:
#Renaming the Column name
df1 = data_csv.withColumnRenamed("customer_name", "customer")
df1.show(5)

#Typecasting Quantity column into int
df1 = data_csv.withColumnRenamed("customer_name", "customer")
df1.show(5)

+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+
|order_id|order_date|customer|region|       category|quantity|unit_price|  sales|discount|returned|
+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+
|       1|2025-06-13|     Eva| South|    Electronics|       2|     49.99|  99.98|       5|     Yes|
|       2|2025-06-23|    John|  West|    Electronics|       9|     49.99| 449.91|       0|     Yes|
|       3|2025-02-25|    John| North|Office Supplies|       4|    199.99| 799.96|       5|      No|
|       4|2025-02-26|     Eva| North|    Electronics|       8|    199.99|1599.92|      15|      No|
|       5|2025-03-13|   Farah| North|    Electronics|       3|     79.99| 239.97|      15|     Yes|
+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+
only showing top 5 rows
+--------+----------+--------+------+---------------+--------+----------+---

In [0]:
#Adding a new column named net_sales
df2 = df1.withColumn("net_sales", round(col("sales") * (1 - col("discount") / 100), 2))
df2.show(5)

#Adding a new Discount Column 
df3 = df2.withColumn("discount_type", when(col("discount") >= 10, "High") .otherwise("Low"))
df3.show(5)

#Also a constant Column using lit
df4 = df3.withColumn("company", lit("Celebal Technologies"))
df4.show(5)

+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+---------+
|order_id|order_date|customer|region|       category|quantity|unit_price|  sales|discount|returned|net_sales|
+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+---------+
|       1|2025-06-13|     Eva| South|    Electronics|       2|     49.99|  99.98|       5|     Yes|    94.98|
|       2|2025-06-23|    John|  West|    Electronics|       9|     49.99| 449.91|       0|     Yes|   449.91|
|       3|2025-02-25|    John| North|Office Supplies|       4|    199.99| 799.96|       5|      No|   759.96|
|       4|2025-02-26|     Eva| North|    Electronics|       8|    199.99|1599.92|      15|      No|  1359.93|
|       5|2025-03-13|   Farah| North|    Electronics|       3|     79.99| 239.97|      15|     Yes|   203.97|
+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+---------+
only showi

In [0]:
#Dropping the constant column
df5 = df4.drop("company")
df5.show(5)

+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+---------+-------------+
|order_id|order_date|customer|region|       category|quantity|unit_price|  sales|discount|returned|net_sales|discount_type|
+--------+----------+--------+------+---------------+--------+----------+-------+--------+--------+---------+-------------+
|       1|2025-06-13|     Eva| South|    Electronics|       2|     49.99|  99.98|       5|     Yes|    94.98|          Low|
|       2|2025-06-23|    John|  West|    Electronics|       9|     49.99| 449.91|       0|     Yes|   449.91|          Low|
|       3|2025-02-25|    John| North|Office Supplies|       4|    199.99| 799.96|       5|      No|   759.96|          Low|
|       4|2025-02-26|     Eva| North|    Electronics|       8|    199.99|1599.92|      15|      No|  1359.93|         High|
|       5|2025-03-13|   Farah| North|    Electronics|       3|     79.99| 239.97|      15|     Yes|   203.97|         High|
+-------

In [0]:
#Fill NA values
filled = data_csv.fillna({"discount": 0, "sales": 0, "unit_price": 0})
filled.show(5)

#Dropping na column
dropped = filled.dropna(how="any", subset=["customer_name"])
dropped.show(5)

+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
|order_id|order_date|customer_name|region|       category|quantity|unit_price|  sales|discount|returned|
+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
|       1|2025-06-13|          Eva| South|    Electronics|       2|     49.99|  99.98|       5|     Yes|
|       2|2025-06-23|         John|  West|    Electronics|       9|     49.99| 449.91|       0|     Yes|
|       3|2025-02-25|         John| North|Office Supplies|       4|    199.99| 799.96|       5|      No|
|       4|2025-02-26|          Eva| North|    Electronics|       8|    199.99|1599.92|      15|      No|
|       5|2025-03-13|        Farah| North|    Electronics|       3|     79.99| 239.97|      15|     Yes|
+--------+----------+-------------+------+---------------+--------+----------+-------+--------+--------+
only showing top 5 rows
+--------+----------+----------

In [0]:
#shuffle(wide Transformation)
summary = dropped.groupBy("region").agg(sum("sales").alias("total_sales"))
summary.show(5)

+------+------------------+
|region|       total_sales|
+------+------------------+
| South|          22508.36|
|  West| 7759.359999999999|
| North|          19758.78|
|  East|19988.690000000002|
+------+------------------+



In [0]:
#aggregates sorted on descending order of the highest sales
summary = dropped.groupBy("category").agg(
    count("*").alias("orders"),
    avg("sales").alias("average_sales"),
    max("sales").alias("highest_sale"),
    min("sales").alias("lowest_sale")
)
summary.orderBy(col("highest_sale").desc()).show()

+---------------+------+-----------------+------------+-----------+
|       category|orders|    average_sales|highest_sale|lowest_sale|
+---------------+------+-----------------+------------+-----------+
|Office Supplies|    37|697.2486486486486|      2999.9|        0.0|
|    Electronics|    28|577.8139285714286|     2699.91|        0.0|
|      Furniture|    35|801.0914285714285|     2399.92|        0.0|
+---------------+------+-----------------+------------+-----------+



In [0]:
#A complete pipeline (read → transform → filter → write)
pipeline = (
    data_csv
    .fillna({"discount": 0, "sales": 0})
    .filter(col("quantity") >= 5)
    .withColumn("net_sales", round(col("sales") * (1 - col("discount") / 100), 2))
    .groupBy("region", "category")
    .agg(sum("net_sales").alias("total_sales"))
)
pipeline.show()

+------+---------------+------------------+
|region|       category|       total_sales|
+------+---------------+------------------+
|  West|    Electronics|           2149.76|
| North|    Electronics|           6494.67|
| North|Office Supplies|           5489.68|
| South|      Furniture|           8239.49|
| South|Office Supplies|           6374.52|
|  West|Office Supplies|           3257.85|
|  East|Office Supplies|2844.6899999999996|
|  East|    Electronics|           1799.87|
| North|      Furniture|1664.8600000000001|
|  East|      Furniture|          11806.86|
| South|    Electronics|3681.1500000000005|
|  West|      Furniture|           1247.84|
+------+---------------+------------------+



In [0]:
#overwriting the data into the csv file
pipeline.write.mode("overwrite").saveAsTable("workspace.default.sales_summary_csv")

#Saving into Paraquet Format
pipeline.write \
    .mode("overwrite") \
    .parquet("/tmp/sales_summary_parquet")